[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/notebooks/factor_screen.ipynb)

# Multi-Factor Fundamental Screen — S&P 500

Screen the S&P 500 by a composite fundamental score across three factors:

- **Quality** — Return on Equity (`NetIncome / StockholdersEquity`)
- **Growth** — Year-over-year revenue growth
- **Efficiency** — Asset turnover (`TotalRevenue / TotalAssets`)

Each factor is z-scored within the universe, then averaged to produce a composite rank. This is the skeleton every quant factor strategy starts from — swap in your own factors, add more, re-weight, layer on a sector neutraliser.

---

In [ ]:
!pip install valuein-sdk -q

In [ ]:
try:
    import os

    from google.colab import userdata

    os.environ["VALUEIN_API_KEY"] = userdata.get("VALUEIN_API_KEY")
except ImportError:
    pass

## 1. Pull latest + prior-year 10-K fundamentals (S&P 500)

Two `LATERAL` subqueries: the first picks the most recent 10-K per company, the second picks the 10-K filed strictly before it. Then we pivot five `standard_concept` values in a single `fact` scan per filing.

Starting from `references` (denormalized flat join) means zero 3-table joins across the full universe.

In [ ]:
import pandas as pd

from valuein_sdk import ValueinClient

client = ValueinClient(tables=["references", "filing", "fact"])

df = client.query("""
    WITH latest AS (
        SELECT
            r.cik, r.symbol, r.name, r.sector,
            f.accession_id, f.filing_date, f.report_date
        FROM "references" r
        JOIN LATERAL (
            SELECT accession_id, filing_date, report_date
            FROM   filing
            WHERE  entity_id = r.cik AND form_type = '10-K'
            ORDER  BY filing_date DESC
            LIMIT  1
        ) f ON TRUE
        WHERE r.is_sp500 = TRUE AND r.is_active = TRUE
    ),
    prior AS (
        SELECT l.cik, f.accession_id AS prior_accession_id
        FROM latest l
        JOIN LATERAL (
            SELECT accession_id
            FROM   filing
            WHERE  entity_id = l.cik
              AND  form_type = '10-K'
              AND  filing_date < l.filing_date
            ORDER  BY filing_date DESC
            LIMIT  1
        ) f ON TRUE
    ),
    latest_facts AS (
        SELECT fa.accession_id,
            MAX(CASE WHEN fa.standard_concept = 'TotalRevenue'       THEN fa.numeric_value END) AS revenue,
            MAX(CASE WHEN fa.standard_concept = 'NetIncome'          THEN fa.numeric_value END) AS net_income,
            MAX(CASE WHEN fa.standard_concept = 'StockholdersEquity' THEN fa.numeric_value END) AS equity,
            MAX(CASE WHEN fa.standard_concept = 'TotalAssets'        THEN fa.numeric_value END) AS total_assets
        FROM fact fa
        WHERE fa.standard_concept IN
              ('TotalRevenue','NetIncome','StockholdersEquity','TotalAssets')
          AND fa.fiscal_period = 'FY'
        GROUP BY fa.accession_id
    ),
    prior_facts AS (
        SELECT fa.accession_id,
            MAX(CASE WHEN fa.standard_concept = 'TotalRevenue' THEN fa.numeric_value END) AS revenue
        FROM fact fa
        WHERE fa.standard_concept = 'TotalRevenue'
          AND fa.fiscal_period = 'FY'
        GROUP BY fa.accession_id
    )
    SELECT
        l.symbol, l.name, l.sector,
        lf.revenue       AS revenue,
        lf.net_income    AS net_income,
        lf.equity        AS equity,
        lf.total_assets  AS total_assets,
        pf.revenue       AS revenue_prior
    FROM latest l
    JOIN latest_facts lf ON lf.accession_id = l.accession_id
    LEFT JOIN prior p    ON p.cik           = l.cik
    LEFT JOIN prior_facts pf ON pf.accession_id = p.prior_accession_id
""")

print(f"Rows returned: {len(df)}")
df.head()

## 2. Compute the factors

Every ratio uses `.where(denominator > 0)` — the NULLIF-equivalent in pandas. Companies with zero or negative denominators simply drop out of that factor rather than producing bogus numbers.

In [ ]:
df["roe"] = df["net_income"] / df["equity"].where(df["equity"] > 0)
df["asset_turnover"] = df["revenue"] / df["total_assets"].where(df["total_assets"] > 0)
df["rev_growth"] = (df["revenue"] - df["revenue_prior"]) / df["revenue_prior"].where(
    df["revenue_prior"] > 0
)

factors = ["roe", "asset_turnover", "rev_growth"]
ranked = df.dropna(subset=factors).copy()


def zscore(s: pd.Series) -> pd.Series:
    # Winsorise at 1st/99th percentile to blunt outliers before z-scoring.
    lo, hi = s.quantile([0.01, 0.99])
    s = s.clip(lower=lo, upper=hi)
    return (s - s.mean()) / s.std(ddof=0)


for f in factors:
    ranked[f"z_{f}"] = zscore(ranked[f])

ranked["composite"] = ranked[[f"z_{f}" for f in factors]].mean(axis=1)
ranked[factors + ["composite"]].describe().round(3)

## 3. Top and bottom of the screen

In [ ]:
cols = ["symbol", "name", "sector", "roe", "rev_growth", "asset_turnover", "composite"]
print("Top 15 by composite factor score")
ranked.nlargest(15, "composite")[cols].round(3)

In [ ]:
print("Bottom 15 by composite factor score")
ranked.nsmallest(15, "composite")[cols].round(3)

## 4. Composite score by sector

In [ ]:
by_sector = (
    ranked.groupby("sector")["composite"]
    .median()
    .sort_values(ascending=False)
    .round(3)
    .to_frame("median_composite")
)
by_sector

## Next steps

- [`dcf_inputs.ipynb`](dcf_inputs.ipynb) — assemble the raw inputs for your own DCF
- [`earnings_momentum.ipynb`](earnings_momentum.ipynb) — YoY revenue/earnings acceleration screen
- [`pit_backtest.ipynb`](pit_backtest.ipynb) — apply these factors with proper PIT discipline